In [1]:
pip install pandas nltk tabulate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from tabulate import tabulate

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('wordnet')

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered)

def label_sentiment(compound_score):
    if compound_score >= 0.05:
        return 'Positive'
    elif compound_score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def main():
    print("\n📄 Sentiment Analysis Dashboard")
    filename = input("Enter your CSV file name (with .csv extension): ")

    try:
        df = pd.read_csv(filename)
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return

    print(f"✅ File '{filename}' loaded successfully. Columns: {list(df.columns)}")

    choice = input("Do you want to perform sentiment analysis on this file? (yes/no): ").strip().lower()
    if choice != 'yes':
        print("Exiting without sentiment analysis.")
        return

    text_col = input("Enter the name of the text column to analyze (e.g., text or comment): ").strip()
    if text_col not in df.columns:
        print(f"❌ Column '{text_col}' not found.")
        return

    print("🧹 Cleaning text...")
    df['cleaned_text'] = df[text_col].apply(clean_text)

    print("🔍 Running VADER Sentiment Analysis...")
    sia = SentimentIntensityAnalyzer()
    df['vader_scores'] = df['cleaned_text'].apply(sia.polarity_scores)
    df['compound'] = df['vader_scores'].apply(lambda x: x['compound'])
    df['positive'] = df['vader_scores'].apply(lambda x: x['pos'])
    df['neutral']  = df['vader_scores'].apply(lambda x: x['neu'])
    df['negative'] = df['vader_scores'].apply(lambda x: x['neg'])
    df['vader_sentiment'] = df['compound'].apply(label_sentiment)
    df.drop(columns=['vader_scores'], inplace=True)

    # Create the final sentiment table
    sentiment_table = df[['cleaned_text', 'compound', 'positive', 'negative', 'neutral', 'vader_sentiment']].head(300)

    print("\n📊 First 300 Rows of Sentiment Table:")
    print(tabulate(sentiment_table, headers='keys', tablefmt='grid', showindex=False))

    # Save the output table to CSV
    sentiment_table.to_csv("Sentiment_Report_3.csv", index=False, encoding='utf-8-sig')
    print("✅ Saved top 300 sentiment rows to 'Sentiment_Report_3.csv'")

    print("\n✅ Program finished. Thank you!")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adittsingh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adittsingh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/adittsingh/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adittsingh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



📄 Sentiment Analysis Dashboard


Enter your CSV file name (with .csv extension):  audio3.csv


✅ File 'audio3.csv' loaded successfully. Columns: ['start', 'end', 'text']


Do you want to perform sentiment analysis on this file? (yes/no):  yes
Enter the name of the text column to analyze (e.g., text or comment):  text


🧹 Cleaning text...
🔍 Running VADER Sentiment Analysis...

📊 First 300 Rows of Sentiment Table:
+-------------------------------------------------------------------+------------+------------+------------+-----------+-------------------+
| cleaned_text                                                      |   compound |   positive |   negative |   neutral | vader_sentiment   |
+===================================================================+============+============+============+===========+===================+
| transcribed turboscribeai go unlimited remove message call centre |     0      |      0     |      0     |     1     | Neutral           |
+-------------------------------------------------------------------+------------+------------+------------+-----------+-------------------+
| handling rude customers                                           |    -0.4588 |      0     |      0.6   |     0.4   | Negative          |
+----------------------------------------------------------